# Compare Experiment Results

Compare each experiment across two versions: `results` vs `results_thesis`

In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict

# Base paths
V1_PATH = Path("results")
V2_PATH = Path("results_thesis")
SEEDS = ["seed123", "seed42", "seed456"]

def load_json(file_path):
    """Load JSON file, return None if not found."""
    try:
        if file_path.exists():
            return json.load(open(file_path))
    except:
        pass
    return None

def get_experiments():
    """Get all experiment names."""
    v1_exps = set(d.name for d in V1_PATH.iterdir() if d.is_dir())
    v2_exps = set(d.name for d in V2_PATH.iterdir() if d.is_dir())
    return sorted(v1_exps & v2_exps)  # Common experiments

experiments = get_experiments()
print(f"Found {len(experiments)} experiments in both versions:")
for exp in experiments:
    print(f"  - {exp}")

Found 7 experiments in both versions:
  - independent_experts
  - langid_router
  - moe_hard
  - moe_soft
  - multitask_lora
  - sequential_forgetting
  - sequential_forgetting_full


In [2]:
def compare_single_exp(exp_name):
    """Compare a single experiment across both versions."""
    print(f"\n{exp_name.upper()}")
    print("-" * 60)
    
    for seed in SEEDS:
        v1_res = load_json(V1_PATH / exp_name / seed / "results.json")
        v2_res = load_json(V2_PATH / exp_name / seed / "results.json")
        v1_pred = load_json(V1_PATH / exp_name / seed / "predictions.json")
        v2_pred = load_json(V2_PATH / exp_name / seed / "predictions.json")
        
        v1_ok = v1_res is not None and v1_pred is not None
        v2_ok = v2_res is not None and v2_pred is not None
        
        print(f"\n{seed}: V1={'OK' if v1_ok else 'MISS'}  V2={'OK' if v2_ok else 'MISS'}")
        
        if isinstance(v1_res, dict) or isinstance(v2_res, dict):
            v1_keys = set(v1_res.keys()) if isinstance(v1_res, dict) else set()
            v2_keys = set(v2_res.keys()) if isinstance(v2_res, dict) else set()
            all_keys = sorted(v1_keys | v2_keys)
            
            for key in all_keys:
                v1_val = v1_res.get(key) if isinstance(v1_res, dict) else None
                v2_val = v2_res.get(key) if isinstance(v2_res, dict) else None
                
                if isinstance(v1_val, dict) or isinstance(v2_val, dict):
                    v1_metrics = set(v1_val.keys()) if isinstance(v1_val, dict) else set()
                    v2_metrics = set(v2_val.keys()) if isinstance(v2_val, dict) else set()
                    all_metrics = sorted(v1_metrics | v2_metrics)
                    
                    if all_metrics:
                        print(f"  {key}:")
                        for metric in all_metrics:
                            v1_m = v1_val.get(metric) if isinstance(v1_val, dict) else None
                            v2_m = v2_val.get(metric) if isinstance(v2_val, dict) else None
                            
                            v1_str = f"{v1_m:.4f}" if isinstance(v1_m, float) else str(v1_m) if v1_m is not None else "-"
                            v2_str = f"{v2_m:.4f}" if isinstance(v2_m, float) else str(v2_m) if v2_m is not None else "-"
                            
                            if isinstance(v1_m, (int, float)) and isinstance(v2_m, (int, float)):
                                diff = v2_m - v1_m
                                rel_diff = abs(diff / v1_m * 100) if v1_m != 0 else 0
                                is_reasonable = abs(diff) < 0.01 or rel_diff < 2
                                status = "OK" if is_reasonable else "DIFF"
                                diff_str = f"{diff:+.4f}"
                                print(f"    {metric:20} V1: {v1_str:12} V2: {v2_str:12} Diff: {diff_str:10} [{status}]")
                            else:
                                match = "=" if v1_str == v2_str else "!="
                                print(f"    {metric:20} V1: {v1_str:12} V2: {v2_str:12} [{match}]")

print("Function created")

Function created


## Compare Individual Experiments

Run each cell below to compare a specific experiment.

In [3]:
compare_single_exp("independent_experts")


INDEPENDENT_EXPERTS
------------------------------------------------------------

seed123: V1=OK  V2=OK
  de:
    bleu                 V1: 0.5812       V2: 0.5807       Diff: -0.0005    [OK]
    bleu_1               V1: 0.7758       V2: 0.7759       Diff: +0.0001    [OK]
    bleu_2               V1: 0.6208       V2: 0.6212       Diff: +0.0004    [OK]
    bleu_3               V1: 0.5242       V2: 0.5235       Diff: -0.0006    [OK]
    bleu_4               V1: 0.4520       V2: 0.4507       Diff: -0.0014    [OK]
    chrf                 V1: 76.5301      V2: 76.5416      Diff: +0.0116    [OK]
    comet                V1: 0.9217       V2: 0.9217       Diff: +0.0000    [OK]
    comet_std            V1: 0.0624       V2: 0.0617       Diff: -0.0006    [OK]
    fta                  V1: 0.7470       V2: 0.7611       Diff: +0.0142    [OK]
    fta_coverage         V1: 0.3993       V2: 0.3993       Diff: +0.0000    [OK]
    fta_mean_sentence    V1: 0.7459       V2: 0.7472       Diff: +0.0013    [OK

In [4]:
compare_single_exp("langid_router")


LANGID_ROUTER
------------------------------------------------------------

seed123: V1=OK  V2=OK
  de:
    bleu                 V1: 0.5704       V2: 0.5767       Diff: +0.0063    [OK]
    bleu_1               V1: 0.7703       V2: 0.7730       Diff: +0.0027    [OK]
    bleu_2               V1: 0.6112       V2: 0.6167       Diff: +0.0055    [OK]
    bleu_3               V1: 0.5127       V2: 0.5197       Diff: +0.0069    [OK]
    bleu_4               V1: 0.4386       V2: 0.4466       Diff: +0.0080    [OK]
    chrf                 V1: 75.7282      V2: 76.1174      Diff: +0.3892    [OK]
    comet                V1: 0.9191       V2: -            [!=]
    comet_std            V1: 0.0629       V2: -            [!=]
    fta                  V1: 0.7105       V2: 0.7490       Diff: +0.0385    [DIFF]
    fta_coverage         V1: 0.3993       V2: 0.3993       Diff: +0.0000    [OK]
    fta_mean_sentence    V1: 0.7106       V2: 0.7350       Diff: +0.0244    [DIFF]
    fta_sentences        V1: 694  

In [5]:
compare_single_exp("moe_hard")


MOE_HARD
------------------------------------------------------------

seed123: V1=OK  V2=OK
  de:
    bleu                 V1: 0.5709       V2: 0.5690       Diff: -0.0019    [OK]
    bleu_1               V1: 0.7708       V2: 0.7685       Diff: -0.0023    [OK]
    bleu_2               V1: 0.6117       V2: 0.6099       Diff: -0.0018    [OK]
    bleu_3               V1: 0.5134       V2: 0.5117       Diff: -0.0017    [OK]
    bleu_4               V1: 0.4390       V2: 0.4371       Diff: -0.0020    [OK]
    chrf                 V1: 75.7852      V2: 75.5820      Diff: -0.2032    [OK]
    comet                V1: 0.9197       V2: 0.9182       Diff: -0.0015    [OK]
    comet_std            V1: 0.0618       V2: 0.0639       Diff: +0.0022    [OK]
    fta                  V1: 0.7328       V2: 0.7409       Diff: +0.0081    [OK]
    fta_coverage         V1: 0.3993       V2: 0.3993       Diff: +0.0000    [OK]
    fta_mean_sentence    V1: 0.7265       V2: 0.7170       Diff: -0.0095    [OK]
    fta_s

In [6]:
compare_single_exp("moe_soft")


MOE_SOFT
------------------------------------------------------------

seed123: V1=OK  V2=OK
  de:
    bleu                 V1: 0.0006       V2: 0.5561       Diff: +0.5555    [DIFF]
    bleu_1               V1: 0.0077       V2: 0.7636       Diff: +0.7559    [DIFF]
    bleu_2               V1: 0.0009       V2: 0.5992       Diff: +0.5983    [DIFF]
    bleu_3               V1: 0.0003       V2: 0.4977       Diff: +0.4974    [DIFF]
    bleu_4               V1: 0.0001       V2: 0.4225       Diff: +0.4224    [DIFF]
    chrf                 V1: 5.3390       V2: 74.9334      Diff: +69.5943   [DIFF]
    comet                V1: 0.1923       V2: 0.9186       Diff: +0.7263    [DIFF]
    comet_std            V1: 0.0900       V2: 0.0624       Diff: -0.0276    [DIFF]
    fta                  V1: 0.0071       V2: 0.7743       Diff: +0.7672    [DIFF]
    fta_coverage         V1: 0.3993       V2: 0.3993       Diff: +0.0000    [OK]
    fta_mean_sentence    V1: 0.0094       V2: 0.7398       Diff: +0.7304

In [7]:
compare_single_exp("multitask_lora")


MULTITASK_LORA
------------------------------------------------------------

seed123: V1=OK  V2=OK
  de:
    bleu                 V1: 0.6049       V2: 0.6050       Diff: +0.0001    [OK]
    bleu_1               V1: 0.7934       V2: 0.7927       Diff: -0.0007    [OK]
    bleu_2               V1: 0.6447       V2: 0.6446       Diff: -0.0001    [OK]
    bleu_3               V1: 0.5507       V2: 0.5503       Diff: -0.0005    [OK]
    bleu_4               V1: 0.4802       V2: 0.4788       Diff: -0.0015    [OK]
    chrf                 V1: 77.8854      V2: 77.9381      Diff: +0.0526    [OK]
    comet                V1: 0.9240       V2: 0.9236       Diff: -0.0004    [OK]
    comet_std            V1: 0.0625       V2: 0.0629       Diff: +0.0003    [OK]
    fta                  V1: 0.7308       V2: 0.7156       Diff: -0.0152    [DIFF]
    fta_coverage         V1: 0.3993       V2: 0.3993       Diff: +0.0000    [OK]
    fta_mean_sentence    V1: 0.7219       V2: 0.7165       Diff: -0.0054    [OK]
 

In [8]:
compare_single_exp("sequential_forgetting")


SEQUENTIAL_FORGETTING
------------------------------------------------------------

seed123: V1=OK  V2=MISS
  de:
    bleu                 V1: 0.5961       V2: -            [!=]
    bleu_1               V1: 0.7876       V2: -            [!=]
    bleu_2               V1: 0.6359       V2: -            [!=]
    bleu_3               V1: 0.5408       V2: -            [!=]
    bleu_4               V1: 0.4692       V2: -            [!=]
    chrf                 V1: 77.3068      V2: -            [!=]
    comet                V1: 0.9223       V2: -            [!=]
    comet_std            V1: 0.0646       V2: -            [!=]
    fta                  V1: 0.7034       V2: -            [!=]
    fta_coverage         V1: 0.3993       V2: -            [!=]
    fta_mean_sentence    V1: 0.7021       V2: -            [!=]
    fta_sentences        V1: 694          V2: -            [!=]
    fta_terms_total      V1: 988          V2: -            [!=]
  en:
    bleu                 V1: 0.6280       V2: -